# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook adheres to best practices for Croissant schema exploration and always references entities using their unique `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\nDataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate the record sets defined in the schema. For each record set, we inspect its fields and list each field's `@id` and human-readable name where available.

In [ ]:
# List record sets and their corresponding fields, referencing by @id
record_sets = []
fields_dict = {}

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for record_set in metadata.recordSet:
        rs_id = record_set['@id'] if isinstance(record_set, dict) and '@id' in record_set else record_set
        record_sets.append(rs_id)
        # Try to fetch fields from the record set
        rs_obj = dataset._lookup(rs_id)
        fields = rs_obj.get('field', []) if rs_obj else []
        fields_list = []
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            # Try to get a field label/name
            fld_obj = dataset._lookup(field_id)
            label = fld_obj.get('name', field_id) if fld_obj and 'name' in fld_obj else field_id
            fields_list.append({'id': field_id, 'label': label})
        fields_dict[rs_id] = fields_list

    print("Record Sets (@id):")
    for rs_id in record_sets:
        print(f"  - {rs_id}")
        if fields_dict.get(rs_id):
            print("    Fields:")
            for field in fields_dict[rs_id]:
                print(f"      - @id: {field['id']}, label: {field['label']}")
else:
    print("No record sets are defined in metadata. Please check the schema or data definition.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, each record set is loaded into a pandas DataFrame, keyed by its `@id`. All entities are referenced by their `@id` as required.

In [ ]:
# Extract records to DataFrames, using the @id
dfs = {}
# For demonstration, use the first available record set
if record_sets:
    for rs_id in record_sets:
        print(f"\nLoading records from record set \"{rs_id}\"...")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"Columns (@id): {list(df.columns)}")
        print(f"Sample records:")
        print(df.head())
else:
    print("No record sets detected for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We select a numeric field by its `@id` and demonstrate filtering, normalization, and grouping by a demographic field (such as gender or education level) referenced by its `@id`.

In [ ]:
# EDA: Filter, normalize, and group
# Choose a record set and numeric field by their @id
if record_sets:
    rs_id = record_sets[0]  # Example: use first record set
    df = dfs[rs_id]
    # Guess a numeric field @id (e.g., GAD-7 total score, present as 'total_gad7')
    candidate_numeric_fields = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or df[col].dtype in [int, float]]
    # Fallback: pick any integer/float column
    if not candidate_numeric_fields:
        candidate_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]

    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by demographic field (such as gender or education)
    group_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower()]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We use matplotlib to visualize the distribution of the selected numeric field and a simple bar plot showing mean normalized scores per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For visualization, use numeric_field_id and group_field_id from EDA
if record_sets:
    df = dfs[record_sets[0]]
    numeric_field = numeric_field_id
    group_field = group_field_id if 'group_field_id' in locals() else None

    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        avg_norm = df.groupby(group_field)[numeric_field].mean()
        avg_norm.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset from Kilifi County, Kenya provides rich information on mental health, demographics, and assessment scores.
- Data loaded via `mlcroissant` allows convenient referencing of record sets and fields by their `@id`.
- Exploratory analysis shows distributions and group-level comparisons for mental health metrics.
- This AI-ready dataset supports further statistical and modeling tasks, promoting FAIR principles in mental health data infrastructure.
